# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [40]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [41]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# TODO: apply sent_tokenize

def protect_acronym_dots(text):
    """
    Protect dots in acronyms (e.g., U.P.C.) by replacing them with a placeholder.
    """
    return re.sub(r'\b([A-Z]\.){2,}(?!\s+[A-Z])', lambda m: m.group().replace('.', '<dot>'), text) # Regex here is used to find the goal sequences fast.

def restore_acronym_dots(text):
    """
    Restore the original dots in acronyms by replacing the placeholder back to a dot.
    """
    return text.replace('<dot>', '.')

sentences = sent_tokenize(protect_acronym_dots(text)) # Splitting the sentences after protecting the acronym dots
sentences = [restore_acronym_dots(sentence) for sentence in sentences] # Restoring the original dots in the sentences
# Sent_tokenize is a robust sentence tokenizer that can handle various punctuation and abbreviations.

print(sentences)


['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O.', 'A report valued the project at $3.2 billion.']


## Q2

In [42]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm

# Regex is ideal for pattern-based text transormtaion

def normalize_text(text):
    """
    Normalize the text by applying the specified transformations:
    """
    
    def money_replacer(match):
        """
        Convert money to words.
        """
        digit_map = {'0':'zero', '1':'one', '2':'two', '3':'three', '4':'four',
                     '5':'five', '6':'six', '7':'seven', '8':'eight', '9':'nine'}
        integer_part = match.group(1) # Splitting the integer and decimal parts
        decimal_part = match.group(2)
        return f"{digit_map[integer_part]} point {digit_map[decimal_part]} billion"
    
    def meters_replacer(match):
        """
        Convert meters to centimeters.
        """
        meters = float(match.group(1))
        cm = int(round(meters * 100))
        return f"{cm} centimeters"

    text = re.sub(r'\b([A-Z]\.){2,}(?!\s+[A-Z])', lambda m: m.group().replace('.', ''), text) # Removing dots in acronyms

    text = re.sub(r'(\d+\.\d+)m\b', meters_replacer, text) # Converting meters to centimeters
    
    text = re.sub(r'\$(\d)\.(\d) billion', money_replacer, text) # Converting money to words
    
    return text

text_norm = normalize_text(text)

print(text_norm)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO. A report valued the project at three point two billion.


## Q3

In [43]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

def lowercasing_text(text):
    """
    Lowercase the text while preserving acronyms, mixed case tokens, and multiword proper nouns with underscores.
    """
    # Step 1: Join multiword proper nouns with underscore
    text = re.sub(r'\b([A-Z][a-z]+)\s+([A-Z][a-z]+)\b', r'\1_\2', text)
    
    tokens = text.split()
    result = []
    prev_ends_sentence = True  # Primera palabra siempre es inicio de frase
    
    for token in tokens:
        match = re.match(r'([^\s,\.!?]+)([\.,!?]*)', token)
        if match:
            word = match.group(1)
            punct = match.group(2)
            
            # Rule 1: Acronyms (ALL CAPS, length > 1) → keep uppercase
            if word.isupper() and len(word) > 1:
                result.append(word + punct)
            # Rule 2: MixedCase → keep as-is
            elif any(c.isupper() for c in word[1:]):
                result.append(word + punct)
            # Rule 3: Multiword proper nouns (underscore) → keep as-is
            elif '_' in word:
                result.append(word + punct)
            # Rule 4: Proper nouns (starts with capital) → keep as-is
            elif word[0].isupper() and len(word) > 1:
                result.append(word + punct)
            # Rule 5: Start of sentence → capitalize
            elif prev_ends_sentence:
                result.append(word.capitalize() + punct)
            # Default: lowercase
            else:
                result.append(word.lower() + punct)
            
            # Update sentence tracker
            prev_ends_sentence = bool(re.search(r'[.!?]', punct))
        else:
            result.append(token)
    
    return ' '.join(result)


text_case = lowercasing_text(text_norm)
print(text_case)



 

In mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO. A report valued the project at three point two billion.


## Q4

In [44]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

nltk.download("punkt_tab")

def tokenize_text(text):
    """
    Tokenize the text using nltk's word_tokenize.
    """
    return word_tokenize(text) # NLTK's word tokenizer

tokens = tokenize_text(text_case)

print(tokens)


['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', '.', 'A', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mariotg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Q5

In [45]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop
from nltk.corpus import stopwords

def remove_stopwords(tokens):
    """
    Removing stopwords from a given text
    """
    stop_words = set(stopwords.words('english')) # Getting the set of English stopwords
    
    tokens_nostop = [] # List to hold tokens that are not stopwords
    
    for token in tokens:
        if token.lower() not in stop_words or token[0].isupper() or '_' in token or token.isdigit(): # Checking if the token is not a stopword or is an entity token
            tokens_nostop.append(token)
    
    return tokens_nostop
    

tokens_nostop = remove_stopwords(tokens)

print(tokens_nostop)


['In', 'mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'He', '186', 'centimeters', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', '.', 'A', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion', '.']


## Q6

In [46]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

def make_bigrams(text):
    """
    Create bigrams from the given text.
    """
    tokens = word_tokenize(text) # Tokenizing the text
    
    bigrams = []
    
    for i in range(len(tokens) - 1):
        bigrams.append((tokens[i], tokens[i + 1])) # Creating bigrams by pairing each token with the next one

    return bigrams

bigrams = make_bigrams(text)

print(bigrams)


[('In', 'mid-February'), ('mid-February', '2026'), ('2026', ','), (',', 'the'), ('the', 'CEO'), ('CEO', 'of'), ('of', 'OpenAI'), ('OpenAI', ','), (',', 'Sam'), ('Sam', 'Altman'), ('Altman', ','), (',', 'visited'), ('visited', 'Barcelona'), ('Barcelona', '.'), ('.', 'He'), ('He', 'is'), ('is', '1.86m'), ('1.86m', 'tall'), ('tall', 'and'), ('and', 'met'), ('met', 'with'), ('with', 'researchers'), ('researchers', 'from'), ('from', 'U.P.C'), ('U.P.C', '.'), ('.', 'and'), ('and', 'U.N.E.S.C.O'), ('U.N.E.S.C.O', '.'), ('.', 'A'), ('A', 'report'), ('report', 'valued'), ('valued', 'the'), ('the', 'project'), ('project', 'at'), ('at', '$'), ('$', '3.2'), ('3.2', 'billion'), ('billion', '.')]


## Q7

In [47]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

tokens = word_tokenize(text) # Tokenize the text

context_counts = defaultdict(int) # Count of how many times each word appears as the first word in a bigram
bigram_counts = defaultdict(lambda: defaultdict(int)) # Count of how many times each bigram (w1, w2) appears

# Count bigrams and contexts
for i in range(len(tokens) - 1):
    current_word = tokens[i]
    next_word = tokens[i + 1]
    
    context_counts[current_word] += 1
    bigram_counts[current_word][next_word] += 1

# Build probability model
model = defaultdict(dict)
for w1 in bigram_counts:
    for w2 in bigram_counts[w1]:
        model[w1][w2] = bigram_counts[w1][w2] / context_counts[w1] # Calculating the probability of w2 given w1


def predict_next(prev_word, model, top_k=3):
    """
    Predict next word given previous word.
    """
    if prev_word not in model:
        return []
    
    next_words = model[prev_word]
    sorted_predictions = sorted(next_words.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_predictions[:top_k]


print(predict_next("OpenAI", model, top_k=3))


[(',', 1.0)]


## Q8

In [48]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = []

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

def get_vocab_from_corpus(corpus):
    """
    Get the vocabulary from the corpus, representing each word as characters + </w>.
    """
    vocab = defaultdict(int)
    for word in corpus.split():
        tokens = tuple(list(word) + ['</w>']) # Representing each word as characters + </w>
        vocab[tokens] += 1 # Counting the frequency of each word in the corpus
    return vocab


def get_pair_frequencies(vocab):
    """
    Compute pair frequencies weighted by word frequency.
    """
    pairs = defaultdict(int)
    
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1]) # Creating pairs of adjacent characters
            pairs[pair] += freq # Incrementing the frequency of the pair by the frequency of the word it appears in
    return pairs


def merge_pair_in_vocab(pair, vocab):
    """
    Merge the most frequent pair in the vocabulary.
    """
    new_vocab = {}
    bigram = ' '.join(pair)  # For string matching
    replacement = ''.join(pair)  # Merged token
    
    for word in vocab:
        word_str = ' '.join(word) # Convert the word tuple to a string for replacement
        new_word_str = word_str.replace(bigram, replacement) # Replace the bigram with the merged token
        new_word = tuple(new_word_str.split()) # Convert back to tuple
        new_vocab[new_word] = vocab[word] # Update the frequency of the new word in the vocabulary
    
    return new_vocab

# Initialize vocabulary
vocab = get_vocab_from_corpus(corpus)
print("Initial vocab:", vocab)

num_merges = 5

for i in range(num_merges):
    pairs = get_pair_frequencies(vocab) # Get the frequency of each pair in the current vocabulary
    
    if not pairs:
        print(f"No more pairs to merge after {i} merges")
        break

    best_pair = max(pairs, key=pairs.get) # Find the most frequent pair
    print(f"Merge {i+1}: {best_pair} (freq: {pairs[best_pair]})")
    
    vocab = merge_pair_in_vocab(best_pair, vocab) # Merge the best pair in the vocabulary
    merges.append(best_pair)

print(merges)


Initial vocab: defaultdict(<class 'int'>, {('l', 'o', 'w', '</w>'): 1, ('l', 'o', 'w', 'e', 'r', '</w>'): 1, ('n', 'e', 'w', 'e', 's', 't', '</w>'): 1, ('w', 'i', 'd', 'e', 's', 't', '</w>'): 1})
Merge 1: ('l', 'o') (freq: 2)
Merge 2: ('lo', 'w') (freq: 2)
Merge 3: ('e', 's') (freq: 2)
Merge 4: ('es', 't') (freq: 2)
Merge 5: ('est', '</w>') (freq: 2)
[('l', 'o'), ('lo', 'w'), ('e', 's'), ('es', 't'), ('est', '</w>')]


## Q9

In [49]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = 45
FP = 5
FN = 10
TN = 6

accuracy = (TP + TN) / (TP + FP + FN + TN)  # Accuracy is the proportion of correct predictions out of all predictions.
precision = TP / (TP + FP)  # Precision is the proportion of true positives out of all positive predictions.
recall = TP / (TP + FN)  # Recall is the proportion of true positives out of all actual positives.
f1 = 2 * (precision * recall) / (precision + recall)  # F1 score is the harmonic mean of precision and recall.

print(accuracy, precision, recall, f1)


0.7727272727272727 0.9 0.8181818181818182 0.8571428571428572
